## 1. Import and setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print("Imported all the necessary modules.")

Imported all the necessary modules.


## 2. Data Ingestion

In [2]:
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
print("Dataset Loaded successfully.")

Dataset Loaded successfully.


## 3. Handling Missing Values and Duplicates

In [3]:
print('Check for Missing values: ')
df.isna().sum()
print('Check for duplicate entries: ')
df.duplicated()
df.shape

Check for Missing values: 
Check for duplicate entries: 


(1470, 35)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

## 4. Dropping the unnecessary columns and splitting the data

In [5]:
from sklearn.model_selection import train_test_split
# Drop unnecessary columns as they are identifiers and do not give any valuable information
df = df.drop(["StandardHours", "EmployeeNumber", "EmployeeCount"], axis=1)
# Define the target variable (y) and the feature matrix (X)
X = df.drop('Attrition', axis=1)
y = df['Attrition']
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("The data split is successful.")

The data split is successful.


## 5. Preparation and transformation of data

In [6]:
from sklearn.compose import ColumnTransformer # to apply different transformations to different columns simultaneously
from sklearn.preprocessing import StandardScaler, OneHotEncoder # for scaling numerical and categorical columns
num_cat_cols = ['EnvironmentSatisfaction', 'Education', 'JobInvolvement', 'JobLevel',
                'JobSatisfaction', 'PerformanceRating', 'RelationshipSatisfaction', 'WorkLifeBalance', 'StockOptionLevel'
] # Leave them as it is

categorical_cols = [
    'BusinessTravel', 'Department', 'EducationField', 'Gender',
    'JobRole', 'MaritalStatus', 'Over18', 'OverTime'
]  # Need to be encoded
numerical_cols = [
    'Age', 'MonthlyIncome', 'MonthlyRate', 'DailyRate', 'HourlyRate',
    'DistanceFromHome', 'TotalWorkingYears', 'YearsAtCompany',
    'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
    'NumCompaniesWorked', 'PercentSalaryHike', 'TrainingTimesLastYear'
] # Need to be scaled



In [7]:
transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)
print("Transformer created successfully.")
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

print(df.shape)

Transformer created successfully.
(1470, 32)


## 6. Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_transformed, y_train)
y_pred_lr = lr.predict(X_test_transformed)
y_prob_lr = lr.predict_proba(X_test_transformed)[:, 1]
print("LOGISTIC REGRESSION")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr, pos_label='Yes'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr, pos_label='Yes'):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_lr, pos_label='Yes'):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")

LOGISTIC REGRESSION
Confusion Matrix:
 [[243  12]
 [ 29  10]]
Accuracy:  0.8605
Precision: 0.4545
Recall:    0.2564
F1-score:  0.3279
ROC-AUC:   0.7047


## 7. Decision Tree

In [9]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_transformed, y_train)
y_pred_dt = dt.predict(X_test_transformed)
y_prob_dt = dt.predict_proba(X_test_transformed)[:, 1]
print("DECISION TREE")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt, pos_label='Yes'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt, pos_label='Yes'):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_dt, pos_label='Yes'):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_dt):.4f}")

DECISION TREE
Confusion Matrix:
 [[243  12]
 [ 34   5]]
Accuracy:  0.8435
Precision: 0.2941
Recall:    0.1282
F1-score:  0.1786
ROC-AUC:   0.6083


## 8. Random Forest Classifier

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_transformed, y_train)
y_pred_rf = rf.predict(X_test_transformed)
y_prob_rf = rf.predict_proba(X_test_transformed)[:, 1]
print("RANDOM FOREST")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, pos_label='Yes'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, pos_label='Yes'):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_rf, pos_label='Yes'):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}")

RANDOM FOREST
Confusion Matrix:
 [[250   5]
 [ 36   3]]
Accuracy:  0.8605
Precision: 0.3750
Recall:    0.0769
F1-score:  0.1277
ROC-AUC:   0.6933


## 9. Bagging Classifier

In [11]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Initialize and train
bag = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5), n_estimators=50, random_state=42)
bag.fit(X_train_transformed, y_train)

# Generate predictions
y_pred_bag = bag.predict(X_test_transformed)
y_prob_bag = bag.predict_proba(X_test_transformed)[:, 1]

# Output evaluation metrics
print("BAGGING CLASSIFIER")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bag))
print(f"Accuracy:  {accuracy_score(y_test, y_pred_bag):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bag, pos_label='Yes'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_bag, pos_label='Yes'):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_bag, pos_label='Yes'):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_bag):.4f}")

BAGGING CLASSIFIER
Confusion Matrix:
 [[247   8]
 [ 34   5]]
Accuracy:  0.8571
Precision: 0.3846
Recall:    0.1282
F1-score:  0.1923
ROC-AUC:   0.6935


## 10. Summary

In [12]:
metrics = {
    'Logistic Regression': {
        'Accuracy': accuracy_score(y_test, y_pred_lr),
        'Precision': precision_score(y_test, y_pred_lr, pos_label='Yes'),
        'Recall': recall_score(y_test, y_pred_lr, pos_label='Yes'),
        'F1-score': f1_score(y_test, y_pred_lr, pos_label='Yes'),
        'ROC-AUC': roc_auc_score(y_test, y_prob_lr)
    },
    'Decision Tree': {
        'Accuracy': accuracy_score(y_test, y_pred_dt),
        'Precision': precision_score(y_test, y_pred_dt, pos_label='Yes'),
        'Recall': recall_score(y_test, y_pred_dt, pos_label='Yes'),
        'F1-score': f1_score(y_test, y_pred_dt, pos_label='Yes'),
        'ROC-AUC': roc_auc_score(y_test, y_prob_dt)
    },
    'Random Forest': {
        'Accuracy': accuracy_score(y_test, y_pred_rf),
        'Precision': precision_score(y_test, y_pred_rf, pos_label='Yes'),
        'Recall': recall_score(y_test, y_pred_rf, pos_label='Yes'),
        'F1-score': f1_score(y_test, y_pred_rf, pos_label='Yes'),
        'ROC-AUC': roc_auc_score(y_test, y_prob_rf)
    },
    'Bagging Classifier': {
        'Accuracy': accuracy_score(y_test, y_pred_bag),
        'Precision': precision_score(y_test, y_pred_bag, pos_label='Yes'),
        'Recall': recall_score(y_test, y_pred_bag, pos_label='Yes'),
        'F1-score': f1_score(y_test, y_pred_bag, pos_label='Yes'),
        'ROC-AUC': roc_auc_score(y_test, y_prob_bag)
    }
}

summary_df = pd.DataFrame(metrics).T
summary_df.index.name = 'Model'
display(summary_df.round(4))


,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.8605,0.4545,0.2564,0.3279,0.7047
Decision Tree,0.8435,0.2941,0.1282,0.1786,0.6083
Random Forest,0.8605,0.3750,0.0769,0.1277,0.6933
Bagging Classifier,0.8571,0.3846,0.1282,0.1923,0.6935


According the outputs of several models, I recommend using logistic regression model to the HR team as there is more accuracy, precision and other metric scores. As logistic regression is a linear model, we can extract it's coefficients weights and we can give reasoning for the prediction made by this model without much effort. Logistic Regression is computationally lightweight. It requires minimal memory and processing power, making it trivial to deploy into a simple web app, dashboard, or even a scheduled script running inside an Excel macro or internal HR tool. Because it has fewer degrees of freedom compared to deep trees or ensembles, it avoids overfitting on this relatively small dataset (~1,470 records), ensuring stable predictions on future hiring cohorts. 

While Logistic Regression generalizes well, all the models tested struggle somewhat with Recall (capturing all actual leavers, sitting around 34% for Logistic Regression and dropping as low as 10% for Random Forest). This is a direct consequence of class imbalance (most employees stay, so the models are naturally biased toward the majority class). Techniques like SMOTE (Synthetic Minority Over-sampling Technique) or adjusting classification probability thresholds in a future iteration would be helpful  to squeeze out extra percentage points in Precision, Recall, and F1-score.

In [13]:
import joblib

# Save the trained Logistic Regression model
joblib.dump(lr, 'logistic_regression_model.pkl')

# Save the column transformer (critical for preprocessing new inputs!)
joblib.dump(transformer, 'column_transformer.pkl')

print("Model and transformer exported successfully!")


Model and transformer exported successfully!
